# Train the lightweight stem separator on Colab (Path B)

From-scratch training of the compact spectrogram-masking U-Net on MUSDB18.
Read `stem_separator/README.md` first — **Path A (fine-tuning pretrained
Demucs on your own multitrack recordings) is the recommended route if you
have isolated stems of your own**; this notebook is Path B, for training a
smaller model from zero on the public dataset instead.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better —
this one benefits more from a faster GPU than the note-detector does).

**Time/disk expectations:** MUSDB18 is ~4.7GB compressed (the standard
`.stem.mp4` release) / ~30GB as unpacked stems. A full training run is
realistically many hours even on a T4 — Colab free tier disconnects after a
period of inactivity/after ~12h, so plan to checkpoint to Google Drive (cell
below) and resume across multiple sessions rather than expecting one sitting
to finish.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — go enable a GPU runtime")

## 1. Mount Google Drive (for the dataset + checkpoints to survive across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/ichi_stem_training"
os.makedirs(DRIVE_ROOT, exist_ok=True)

## 2. Get the training code

Same two options as the note-detector notebook — clone your repo or upload
the folder directly.

In [ ]:
# Option A: clone your private repo
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # replace before running
!git clone https://{GITHUB_TOKEN}@github.com/Zetthilly/Ichi.git
%cd Ichi/training/stem_separator

```python
# Option B: manual upload instead — same pattern as the note-detector notebook
# from google.colab import files
# import zipfile, io
# uploaded = files.upload()
# ...
```

## 3. Install dependencies

In [ ]:
!pip install -q torch torchaudio musdb museval ai-edge-torch

## 4. Get MUSDB18

Download once, then keep it on Drive so you don't re-download every session.
The dataset is hosted on Zenodo — check
[sigsep.github.io/datasets/musdb](https://sigsep.github.io/datasets/musdb.html)
for the current download link before running this, since dataset hosting
URLs do sometimes change.

In [ ]:
MUSDB_DIR = f"{DRIVE_ROOT}/musdb18"

if not os.path.exists(MUSDB_DIR):
    os.makedirs(MUSDB_DIR, exist_ok=True)
    # Replace this URL with the current one from sigsep.github.io/datasets/musdb.html
    # if it has changed since this notebook was written.
    !wget -O /content/musdb18.zip "https://zenodo.org/record/1117372/files/musdb18.zip"
    !unzip -q /content/musdb18.zip -d {MUSDB_DIR}
else:
    print("MUSDB18 already present on Drive, skipping download.")

## 5. Train

Checkpoints save every epoch straight to Drive (`--checkpoint-path`) so a
disconnected Colab session doesn't lose progress — just re-run this cell to
continue; `torch.load`-based resume isn't wired into the script yet, so for
now treat an interrupted run as a fresh one unless you add that yourself
(`lightweight_unet.py` is a good, small place to add
`--resume-from checkpoint.pt` if you want it).

In [ ]:
!python lightweight_unet.py \
    --musdb-root {MUSDB_DIR} \
    --epochs 50 \
    --batch-size 4 \
    --checkpoint-path {DRIVE_ROOT}/lightweight_unet.pt

## 6. Evaluate (optional but recommended)

`museval` computes standard source-separation metrics (SDR/SIR/SAR) against
MUSDB18's test set — worth running before trusting the model, since training
loss going down doesn't always mean the separation sounds good.

In [ ]:
# A minimal evaluation loop — adapt as needed; museval expects
# reference/estimate audio pairs per track. See museval's docs for the
# exact expected format if this needs adjusting for your output shapes.
import musdb
import museval

mus_test = musdb.DB(root=MUSDB_DIR, subsets="test")
print(f"{len(mus_test.tracks)} test tracks available for evaluation")
# Run your trained model over a few test tracks and compare against
# museval.eval_mus_track(track, estimates) — left as a next step since the
# exact wiring depends on how you want to batch/report this.

## 7. Export to TFLite

**Read the comment block inside `export_to_tflite.py` first** — the model
in `lightweight_unet.py` operates on spectrograms internally, but
`TFLiteStemSeparator.kt` sends/expects raw waveforms, so you need to wrap
the model (wrapper code is provided in that comment block) before
converting.

In [ ]:
!python export_to_tflite.py \
    --checkpoint {DRIVE_ROOT}/lightweight_unet.pt \
    --output stem_separator.tflite \
    --chunk-samples 352800

## 8. Download and integrate

1. Download `stem_separator.tflite` below.
2. Copy it to `app/src/main/assets/models/stem_separator.tflite` in the
   Android project (matching `modelAssetPath` in `TFLiteStemSeparator.kt`,
   or update that constant if you used a different filename).
3. Set `MODEL_CHUNK_SAMPLES` / `MODEL_STEM_COUNT` in `TFLiteStemSeparator.kt`
   to match this model exactly — `MODEL_STEM_COUNT = 4` (drums/bass/other/
   vocals, matching `STEMS` in `lightweight_unet.py`) and
   `MODEL_CHUNK_SAMPLES` matching whatever `--chunk-samples` you exported
   with above.

In [ ]:
from google.colab import files
files.download("stem_separator.tflite")